# Tahoe-100M: repeating the well-level tests on DMSO wells

Tahoe-100M has 14 plates of 96 wells; each well (`sample`) holds about 50 cell lines together, as each Parse well holds all cell types.
Each plate has 2 DMSO wells (28 in total). We use them to repeat three tests from the paper:
1. share of the difference between two control wells due to the well (Figure 2A),
2. DEGs between two identical control wells (Figure 4E),
3. well noise shared across cell lines (Figure 2B).

Run on a machine with internet access. `pip install huggingface_hub pyarrow pandas scipy`

## 1. Setup and sample metadata
Result of the first check: 2 DMSO wells per plate (28 in total). Plates 7 to 9 also list a Trametinib well solvated in DMSO; it is excluded.

In [1]:
import os
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import scipy.sparse as sp
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfFileSystem, list_repo_files

REPO = "tahoebio/Tahoe-100M"
OUT = "cache"; os.makedirs(OUT, exist_ok=True)
F_PB, F_META = f"{OUT}/tahoe_dmso_pb.npz", f"{OUT}/tahoe_dmso_meta.csv"
F_DONE = f"{OUT}/tahoe_dmso_files_done.txt"
SEED = 0

fs = HfFileSystem()
files = list_repo_files(REPO, repo_type="dataset")
data_files = sorted(f for f in files if f.startswith("data/") and f.endswith(".parquet"))
sm = pd.read_parquet(f"hf://datasets/{REPO}/metadata/sample_metadata.parquet", filesystem=fs)
DMSO = sorted(sm.loc[sm["drug"] == "DMSO_TF", "sample"])
PLATE = dict(zip(sm["sample"], sm["plate"]))
print(len(data_files), "data files;", len(DMSO), "DMSO wells")
print(sm[sm["drug"] == "DMSO_TF"].groupby("plate")["sample"].nunique().to_string())

3388 data files; 28 DMSO wells
plate
plate1     2
plate10    2
plate11    2
plate12    2
plate13    2
plate14    2
plate2     2
plate3     2
plate4     2
plate5     2
plate6     2
plate7     2
plate8     2
plate9     2


## 2a. Find the blocks (row groups) that contain DMSO cells
Each file mixes about 60 wells, but cells are sorted by well inside the file, so the DMSO cells of a file sit in one or two of its 29 blocks.
This step reads only the small footer of each file, which stores the first and last `sample` of each block, and keeps the blocks whose range contains a DMSO well.
HuggingFace limits requests (5,000 per 5 minutes): the loop pauses between files and waits when the limit is hit. Log in once with `huggingface-cli login` (free account) for a higher limit.
Results are saved, so the cell can be stopped and run again.

In [2]:
import time, json
from datetime import datetime
F_CAND = f"{OUT}/tahoe_dmso_candidate_blocks.json"

def with_retry(fn, *a, tries=8):
    for t in range(tries):
        try:
            return fn(*a)
        except Exception as e:
            if "429" in str(e) or "Too Many Requests" in str(e):
                wait = 60 * (t + 1); print(f"rate limit, waiting {wait} s", flush=True); time.sleep(wait)
            else:
                raise
    raise RuntimeError("too many retries")

def dmso_blocks(f):
    """Indices of the row groups whose sample range contains a DMSO well (footer only)."""
    with fs.open(f"datasets/{REPO}/{f}", block_size=2**20) as fh:
        md_ = pq.ParquetFile(fh).metadata
        i = md_.schema.names.index("sample")
        out = []
        for rg in range(md_.num_row_groups):
            st = md_.row_group(rg).column(i).statistics
            if st is None or not st.has_min_max or any(st.min <= s <= st.max for s in DMSO):
                out.append(rg)
        return out

print("start:", datetime.now().strftime("%H:%M:%S"))
cand = json.load(open(F_CAND)) if os.path.exists(F_CAND) else {}
for k, f in enumerate(data_files):
    if f in cand:
        continue
    cand[f] = with_retry(dmso_blocks, f)
    time.sleep(0.15)
    if k % 200 == 0:
        json.dump(cand, open(F_CAND, "w"))
        print(f"{k}/{len(data_files)} files, {sum(len(v) for v in cand.values())} blocks with DMSO",
              datetime.now().strftime("%H:%M:%S"), flush=True)
json.dump(cand, open(F_CAND, "w"))
CAND = {f: v for f, v in cand.items() if v}
print(len(CAND), "files,", sum(len(v) for v in CAND.values()), "blocks to read")

start: 23:22:54
0/3388 files, 0 blocks with DMSO 23:22:54
200/3388 files, 241 blocks with DMSO 23:24:30
400/3388 files, 547 blocks with DMSO 23:26:15
600/3388 files, 844 blocks with DMSO 23:28:03
800/3388 files, 1114 blocks with DMSO 23:29:52
1000/3388 files, 1346 blocks with DMSO 23:31:41
1200/3388 files, 1578 blocks with DMSO 23:33:27
1400/3388 files, 1812 blocks with DMSO 23:35:24
1600/3388 files, 2062 blocks with DMSO 23:37:09
1800/3388 files, 2306 blocks with DMSO 23:38:55
2000/3388 files, 2514 blocks with DMSO 23:40:38
2200/3388 files, 2700 blocks with DMSO 23:42:27
2400/3388 files, 2970 blocks with DMSO 23:44:16
2600/3388 files, 3246 blocks with DMSO 23:45:58
2800/3388 files, 3431 blocks with DMSO 23:47:41
3000/3388 files, 3690 blocks with DMSO 23:49:21
3200/3388 files, 3966 blocks with DMSO 23:51:09
1910 files, 4212 blocks to read


## 2b. Read only those blocks and keep the DMSO cells
Expression lists are raw counts. The first entry of each cell (gene token 1, value -2) is a special token and is dropped.
Only the DMSO blocks of each file are downloaded (about 2 of 29). Progress is saved after every file, so the cell can be stopped and run again.

In [3]:
import pickle
F_STATE = f"{OUT}/tahoe_dmso_state.pkl"
COLS = ["genes", "expressions", "sample", "cell_line_id", "BARCODE_SUB_LIB_ID"]
if os.path.exists(F_STATE):
    sums, ncell, first_bc, done = pickle.load(open(F_STATE, "rb"))
else:
    sums, ncell, first_bc, done = {}, {}, {}, set()
rng = np.random.default_rng(SEED + len(done))

def read_blocks(f, blocks):
    with fs.open(f"datasets/{REPO}/{f}", block_size=2**22) as fh:
        t = pq.ParquetFile(fh).read_row_groups(blocks, columns=COLS)
    return t.filter(pc.is_in(t.column("sample"), value_set=pa.array(DMSO))).to_pandas()

print("start:", datetime.now().strftime("%H:%M:%S"))
for k, (f, blocks) in enumerate(CAND.items()):
    if f in done:
        continue
    df = with_retry(read_blocks, f, blocks)
    if len(df):
        half = np.where(rng.random(len(df)) < 0.5, "A", "B")
        for (smp, cl, h), idx in df.groupby([df["sample"], df["cell_line_id"], half]).indices.items():
            g = np.concatenate(df["genes"].values[idx]); x = np.concatenate(df["expressions"].values[idx])
            keep = (g != 1) & (x >= 0)
            row = sp.csr_matrix((x[keep].astype(np.float32), (np.zeros(keep.sum(), int), g[keep])), shape=(1, 70000))
            key = (PLATE[smp], smp, cl, h)
            sums[key] = sums[key] + row if key in sums else row
            ncell[key] = ncell.get(key, 0) + len(idx)
        for smp, bc in zip(df["sample"], df["BARCODE_SUB_LIB_ID"]):
            first_bc.setdefault(smp, set()).add(bc.split("_")[0])
    done.add(f)
    time.sleep(0.3)
    if len(done) % 50 == 0:
        pickle.dump((sums, ncell, first_bc, done), open(F_STATE, "wb"))
        print(f"{len(done)}/{len(CAND)} files, {sum(ncell.values())} DMSO cells",
              datetime.now().strftime("%H:%M:%S"), flush=True)
pickle.dump((sums, ncell, first_bc, done), open(F_STATE, "wb"))
print("done:", sum(ncell.values()), "DMSO cells")

start: 23:52:43
50/1910 files, 56861 DMSO cells 23:53:48
100/1910 files, 128249 DMSO cells 23:55:19
150/1910 files, 195187 DMSO cells 23:56:47
200/1910 files, 256137 DMSO cells 23:58:11
250/1910 files, 317215 DMSO cells 23:59:22
300/1910 files, 378722 DMSO cells 00:00:49
350/1910 files, 445661 DMSO cells 00:02:29
400/1910 files, 508225 DMSO cells 00:04:09
450/1910 files, 558895 DMSO cells 00:05:42
500/1910 files, 591808 DMSO cells 00:06:41
550/1910 files, 624139 DMSO cells 00:07:38
600/1910 files, 678648 DMSO cells 00:08:47
650/1910 files, 740541 DMSO cells 00:10:00
700/1910 files, 801965 DMSO cells 00:11:09
750/1910 files, 856242 DMSO cells 00:12:10
800/1910 files, 907302 DMSO cells 00:13:12
850/1910 files, 965511 DMSO cells 00:14:34
900/1910 files, 1035265 DMSO cells 00:15:57
950/1910 files, 1103412 DMSO cells 00:17:22
1000/1910 files, 1175812 DMSO cells 00:18:52
1050/1910 files, 1251481 DMSO cells 00:20:28
1100/1910 files, 1329683 DMSO cells 00:21:55
1150/1910 files, 1384578 DMSO ce

## 3. Save the pseudobulks

In [4]:
keys = sorted(sums)
M = sp.vstack([sums[k] for k in keys]).tocsr()
used = np.flatnonzero(np.asarray(M.sum(0)).ravel() > 0)
M = M[:, used]
sp.save_npz(F_PB, M)
np.save(f"{OUT}/tahoe_dmso_gene_tokens.npy", used)
meta = pd.DataFrame(keys, columns=["plate", "sample", "cell_line", "half"])
meta["n_cells"] = [ncell[k] for k in keys]
meta.to_csv(F_META, index=False)
print("saved", M.shape, "cells:", meta.n_cells.sum())
print(meta.groupby(["plate", "sample"]).n_cells.sum().to_string())
print("median cells per well and cell line:", meta.groupby(["sample", "cell_line"]).n_cells.sum().median())

saved (2800, 51496) cells: 2205786
plate    sample  
plate1   smp_1589     83959
         smp_1590     87338
plate10  smp_2453    141054
         smp_2454     59984
plate11  smp_2549     84649
         smp_2550     95423
plate12  smp_2645     77980
         smp_2646     99176
plate13  smp_2669     66756
         smp_2670     44884
plate14  smp_2837     40140
         smp_2838     55432
plate2   smp_1685    133902
         smp_1686    111635
plate3   smp_1781     39806
         smp_1782     33927
plate4   smp_1877     73119
         smp_1878     82569
plate5   smp_1973     80088
         smp_1974    102179
plate6   smp_2069     95563
         smp_2070    112874
plate7   smp_2165     37355
         smp_2166     50678
plate8   smp_2261     79224
         smp_2262    118248
plate9   smp_2357     66763
         smp_2358     51081
median cells per well and cell line: 1365.0


## 4. Does the first barcode round identify the well?
In split-pool chemistry the first-round barcode marks the well. One id per DMSO well would let us recover plate positions later.

In [5]:
for s in DMSO[:6]:
    print(s, PLATE[s], sorted(first_bc.get(s, []))[:8])

smp_1589 plate1 ['95']
smp_1590 plate1 ['96']
smp_1685 plate2 ['95']
smp_1686 plate2 ['96']
smp_1781 plate3 ['95']
smp_1782 plate3 ['96']
